# LoRA与QLoRA

## 简介

 LoRA：LoRA 是一种用于微调大型语言模型的技术，通过低秩近似方法降低适应数十亿参数模型（如 GPT-3）到特定任务或领域。 QLoRA：QLoRA 是一种高效的大型语言模型微调方法，它显著降低了内存使用量，同时保持了全 16 位微调的性能。它通过在一个固定的、4 位量化的预训练语言模型中反向传播梯度到低秩适配器来实现这一目标。


![](Image/2025-04-01-16-47-36.png)

## 环境搭建

### 环境要求

1、创建虚拟环境，python==3.11,auto_gptq

2、openwebui

2、LLama Factory

3、modelscope-下载模型Qwen/Qwen2.5-1.5B-Instruct

### 安装modelscope

为了从modelscope下载Qwen/Qwen2.5-1.5B-Instruct，需要先安装modelscope

### 从modelscope上下载Qwen/Qwen2.5-1.5B-Instruct

新建python脚本download.py，代码如下

In [ ]:
#模型下载
from modelscope import snapshot_download
model_dir = snapshot_download('Qwen/Qwen2.5-1.5B-Instruct'
                              , cache_dir="/root/autodl-tmp/model/")

切换到download.py所在目录，执行脚本

![](Image/2025-04-01-22-37-16.png)

![](Image/2025-04-01-22-38-21.png)

### 创建envopenwebui虚拟环境

conda create -n envopenwebui python==3.11 -y

conda activate envopenwebui

pip install open-webui


### 安装LLaMA-Factory

ls

cd autodl-tmp/

ls

git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git

cd LLaMA-Factory

pip install -e .

## 配置数据集

本次采用的是多轮对话的数据集，将上fintech.json和identity.json传到/root/LLaMA-Factory/data目录下

![](Image/2025-04-01-23-01-00.png)

配置数据集

![](Image/2025-04-01-23-02-41.png)

## 启动webui


conda activate llamafactory

cd /root/LLaMA-Factory

llamafactory-cli webui

![](Image/2025-04-01-23-08-49.png)

## webui参数设置

对于“量化等级”，如果模型特别大就使用4，如果模型不是很大就用8，“量化方法”默认的就是最稳定的

![](Image/2025-04-01-23-36-31.png)

对于“加速方式”，主流的是“flashattn2”，但是显卡架构要在sm80以上。如果不确定就选auto

![](Image/2025-04-01-23-16-16.png)

“LoRA 秩”取值在32~128之间，经过大量验证，“LoRA 秩”取64，“LoRA 缩放系数”取128比较合适，二者一般是两倍关系。“训练轮数”可以写大点，等待loss收敛了停下就行。“计算类型”选“bf16",如果设备不支持就选“fp16”，截断长度要根据数据集文本长度来，如果设置太大会占用显存，如果设置太小会导致数据不完整。“批处理大小”根据训练时显存占用情况来，让显存占用90%左右。

![](Image/2025-04-01-23-27-09.png)

![](Image/2025-04-01-23-37-32.png)

缺啥装啥

![](Image/2025-04-01-23-40-09.png)

pip install bitsandbytes

![](Image/2025-04-01-23-41-02.png)

## 调试批次大小

训练已经成功启动了

![](Image/2025-04-01-23-44-54.png)

![](Image/2025-04-01-23-45-10.png)

此时用nvitop查看显存占用情况

![](Image/2025-04-01-23-45-52.png)

显存占用太小，点击“中断”，调整“批处理大小”后重新开始

![](Image/2025-04-01-23-55-46.png)

如果报以下错误就删除训练保存的结果

![](Image/2025-04-01-23-56-42.png)

![](Image/2025-04-01-23-59-13.png) ![](Image/2025-04-01-23-58-53.png)  

如果遇到CUDA out of memory就减小批次大小

![](Image/2025-04-02-00-02-00.png)

### 训练完成标志

![](Image/2025-04-02-01-06-39.png)

### 模型导出

![](Image/2025-04-02-01-11-58.png)

报错

![](Image/2025-04-02-01-13-04.png)

# 大模型转换为 GGUF 

## 什么是 GGUF

GGUF 格式的全名为（GPT-Generated Unified Format），提到GGUF 就不得不提到它的前身 GGML(GPT-Generated Model Language）。GGML 是专门为了机器学习设计的张量库，最早可以追溯到 2022/10。其目的是为了有一个单文件共享的格式，并且易于在不同架构的 GPU 和 CPU 上进行推理。但在后续的开发中，遇到了灵活性不足、相容性及难以维护的问题。


## 为什么要转换 GGUF 格式

GGUF 实际上是基于 GGJT 的格式进行优化的，并解决了 GGML 当初面临的问题，包括：
1）可扩展性：轻松为 GGML 架构下的工具添加新功能，或者向 GGUF 模型添加新 Feature，不会破坏与现有模型的兼容性。
2）对 mmap（内存映射）的兼容性：该模型可以使用 mmap 进行加载（原理解析可见参考），实现快速载入和存储。（从 GGJT 开始导入，可参考 GitHub）
3）易于使用：模型可以使用少量代码轻松加载和存储，无需依赖的 Library，同时对于不同编程语言支持程度也高。
4）模型信息完整：加载模型所需的所有信息都包含在模型文件中，不需要额外编写设置文件。
5）有利于模型量化：GGUF 支持模型量化（4 位、8 位、F16），在 GPU 变得越来越昂贵的情况下，节省 vRAM 成本也非常重要

## 将Hugging face模型转换为GGUF

需要用llama.cpp仓库的convert_hf_to_gguf.py脚本来转换

开启学术加速
source /etc/network_turbo

git clone https://github.com/ggerganov/llama.cpp.git

这个东西对pytorch有要求，要在虚拟环境中装
pip install -r llama.cpp/requirements.txt

![](Image/2025-04-02-00-43-03.png)